# Running Full Simulations

In this workbook, we'll take a quick look at how we can actually run a full simulation, start to finish. The only step we'll skip is the generation of a simualted events file.

First, we'll go to the directory which contains eic-shell. We'll then run eic-shell.

Note that for this tutorial, we will be copy pasting commands from cells to a terminal window.

## Setup

A few cells we'll run to enter eic-shell and set things up

```console
cd ~/eic
```

```console
./eic-shell
```

Once in eic-shell, you should see your terminal prompt preceded by -

jug_dev>

A lot of stuff is pre-installed in eic-shell, but we need to load in the version we want and set some environment variables (e.g. $DETECTOR_PATH).

Luckily, scripts already exist for us to do this, we can run - 

```console
source /opt/detector/epic-main/bin/thisepic.sh
```

to load in the ePIC detector geometry. At this point, we're ready to go.

## Step 1 - Afterburning Files

Strictly speaking, this step is not *neccessary*, but we do need to do it if we want a "realistic" simulation.

Afterburner will incorporate numerous beam effects such as the beam divergence, crabbing and crossing angles. See [this pdf](https://github.com/eic/documents/blob/master/reports/general/Note-Simulations-BeamEffects.pdf) for more info.

Applying the afterburner is relatively easy, we just need to provide it an input .hepmc3 event file. Luckily, we already have one from our git repository:

- Example_Events.hepmc3

Let's see what options are available to us from afterburner. Run:

```console
abconv -h
```

Helpfully, some information is also given to us along with commands we can use to run afterburner. We can run it on our file and see what happens:

```console
cd HSF-India
abconv Example_Events.hepmc3
```

Uh oh! Looks like it isn't happy with something! It seems to be complaining about the beam energy combination:

"10x130 is not a valid energy combination!!"

Afterburner is sort of correct here. 10x130 isn't something that is pre-defined in afterburner, but it is an energy combination we actually expect. It's just a little bit new for our software (this is now one of the "default" beam energy combinations we expect to run in the early science programme).

That information doesn't really help us actually run it on our file though. What *does* is that the 10x100 combination is probably good enough. As such, we can tell afterburner to run with a specific preset:

```console
abconv -p ip6_hiacc_100x10 Example_Events.hepmc3
```

Now afterburner has run without issue. We see some information on the configuration printed to screen. If we do -

```console
ls -lrth
```

We can see that we have two new files -

- ab_output.hepmc
- ab_output.hist.root

The .hepmc files is the one we need going forward. 

### Quick Task

Take a look at our original input file and our new, afterburned file. Convince yourself that afterburner has actually done something!

### Manual Naming

Our file name before was just created automatically. This is fine for now, but if we're running multiple files. This isn't a good idea. Our output will be overwritten if we aren't careful.

As such, it's generally a good idea to name our file. Nice and easy with afterburner -

```console
abconv -p ip6_hiacc_100x10 -o Afterburned_Events Example_Events.hepmc3
```

Note that we still get two outputs. Some output histograms in a root file, and our .hepmc file. We can disable the root file if we really want with another flag. 

## Step 2 - Running the Simulation

Our next step is to actually feed our events into the simulation and run it. In our eic-shell environment, we can run this via -

```console
npsim -h
```

This time, we get an enormous info dump, there are a few arguments we will need -

- --compactFile
    - The detector geometry we will use in our simulation 
- --numberOfEvents
    - The number of events we'll process
- --inputFiles
    - The input .hepmc event file to run
- --outputFile
    - The output file we will save our events to
 
Let's go through these one by one (some are a bit more obvious).

Our compact file argument is used to select the detector geometry to use. We might want to choose to simulate a specific subset of detectors, or a specific version of the full ePIC detector for example. We have several geometries pre-defined which we loaded in earlier. We can check them via -

```console
ls $DETECTOR_PATH
```

Lots of options! We had 10x130 events though, so we'll use - 

- epic_craterlake_10x130.xml

We'll take a closer look at this in a minute too.

numberOfEvents - Pretty self explanatory, how many events we want to process in our simulation. We'll start with 10.

inputFiles - Again, pretty self explanatory, the files with our input events that we want to process.

**Note**, as with all pathing in a terminal, we need to be a bit careful here. Depending upon where we execute npsim, we will need to give **either** the correct *relative* path or the *full* path to the file.

outputFile - Again, pretty self explanatory, the name of the output file we want to create.

So, combining our arguments, we can try to run:

```console
npsim --compactFile $DETECTOR_PATH/epic_craterlake_10x130.xml --numberOfEvents 10 --inputFiles Afterburned_Events.hepmc --outputFile 10on130_10_TestOutput.edm4hep.root
```

**Note** - This command produces a lot of output to screen and might take a little time to process!

Once it's done, we should have a new file with the name we defined in our directory.

### Step 2 - Additional Info

Before, we just picked out a geometry .xml file and ran it. We might want to take a closer look at this geometry file. We can interpret the geometry and spit out a file we can open and explore.

When we get our output, we will open it using the root webviewer - https://eic.phy.anl.gov/geoviewer/ 

To output our geometry, we can run (within eic-shell):

```console
dd_web_display --export $DETECTOR_PATH/epic_craterlake_10x130.xml
```

We should now have a new file:

- detector_geometry.root

We can download this and upload it to the webviewer to take a look, before we do that though, one quick thing.

Let's run a longer simulation and leave it running whilst we look at our geometry and discuss the next step:

```console
npsim --compactFile $DETECTOR_PATH/epic_craterlake_10x130.xml --numberOfEvents 100 --inputFiles Afterburned_Events.hepmc --outputFile 10on130_100_TestOutput.edm4hep.root
```

This one really will take a long time (740s when I tested it - 12 minutes!). Note that the output filename is subtly differet here.

## Step 3 - Reconstruction

Our final step is to reconstruct our simulation output. For this, we will use EICrecon. We've seen the output of EICrecon before, but now we'll generate our own.

Again, we'll start by seeing what arguments we can give EICrecon:

```console
eicrecon -h
```

And once again, a lot of options, we'll be using:

- -Ppodio:outout_file
    - The name of our output podio file (i.e the files we analysed previously)
- Pjana:nevents
    - The number of events we'll run 
- pdd4hep:xml_files
    - The simulation geometry file we used to simulate the file we're using

So, let's try our reconstruction:

```console
eicrecon -Ppodio:output_file=10on130_10_TestRecon.edm4hep.root -Pjana:nevents=10 -Pdd4hep:xml_files=$DETECTOR_PATH/epic_craterlake_10x130.xml 10on130_10_TestOutput.edm4hep.root
```

**This will take some time,** but luckily not as much as the simulation. We'll also have a lot of output to screen.

The file we now have contains output which will look very much like what we used previously. The underlying reaction is quite different though, so when we analyse our file, the output will look quite different.

We can also process our 100 event file (which will again, take some time):

```console
eicrecon -Ppodio:output_file=10on130_100_TestRecon.edm4hep.root -Pjana:nevents=100 -Pdd4hep:xml_files=$DETECTOR_PATH/epic_craterlake_10x130.xml 10on130_100_TestOutput.edm4hep.root
```

We will come back to this.

### Quick Task

Once our 100 event files is done, take a quick look at the contents using uproot. Try to see if you can find and plot our reconstructed and MC truth information for pions in the output.

## Further Thoughts

As we've seen, running the simulation and reconstruction can be very slow. The output of both can also be relatively large. As such, a few thoughts:

- We *can* run simulations in this binderhub, but we probably shouldn't (or even *couldn't* in any meaningful way)
- Better would be to install the eic-shell on our local machine and run it there
    - If we have singularity/apptainer, we can do this quite easily as eic-shell is a container.
- Even running locally though, we'll likely bump into a similar issue to the one we have with the binderhub
- Local resources aren't going to be quite enough

Better still would be to utilise High Performance Computing (HPC) facilities, either at a local institution (maybe your university has a computing cluster?) OR resources at one of the two EIC host labs -

- BNL Scintific Data Computing Centre - [https://www.sdcc.bnl.gov/](SDCC)
- Jefferson Lab Farm - [https://scicomp.jlab.org/scicomp/](Farm)

If you get more involved with the EIC, you might want to look at getting an account for one (or both!) of these facilities to make use of their resources in running your simulations and analysis (or other software)

### Warnings

Finally, a major disclaimer. A lot of the time, you should NOT be starting from scratch and processing through the simulation and reconstruction yourself. There are numerous reasons -

- Computing time intensive
- Versioning errors/mismatch
- Not as reproducible (if you find an error, people will need to try and reproduce it from your environment)

Where possible, use files from official simulation campaigns (bringing us full circle, see the first lesson for using a simulation campaign file in a script!). That being said, for testing and iterating rapidly on a design change, running small jobs yourself may be the way to go. It may also help you to understand the full process by seeing the steps involved.